# Vježba 01d

Proučite dataset `Walmart.csv` i definirajte što je target varijabla. Iz dataseta izbacite sve značajke koje nisu numeričke, a ako negdje nedostaje vrijednost umetnite medijan. Model trenirajte sa `linearnom regresijom`. Iz dataseta izbacite sve značajke koje nisu numeričke, a ako negdje nedostaje vrijednost umetnite medijan. Model trenirajte sa `k-NN-R`. Dataset pripremite sa `train_test_split` funkcijom i pri tom koristite `random_state=21`, a 20% neka vam ostane za testiranje. Nakon toga sa preostalih 80% koristite metodu `cross_val_score` i to na tri razzličita načina koji se razlikuju po vrijednosti cv. U nastavku su navedeni:

- cv=5
- cv=LeaveOneOut()
- cv=ShuffleSplit(test_size=.25)

Za izračun greške koristite `scoring='neg_mean_squared_error'`. Na kraju izračunajte grešku modela (RMSE) sa testnim podacima. Ispišite sve tri srednje vrijednost nizova s greškama (RMSE), ali i grešku (RMSE) sa testnim podacima.

In [21]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn import model_selection
from sklearn.neighbors import KNeighborsRegressor

In [22]:
df = pd.read_csv("Walmart.csv", low_memory=False)
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [23]:
target = df["Weekly_Sales"]
features_num = df.select_dtypes(include=[np.number])

imputer = SimpleImputer(strategy="median")
features_num_imputed = pd.DataFrame(imputer.fit_transform(features_num), columns=features_num.columns)

features = features_num_imputed.drop(columns=["Weekly_Sales"]).select_dtypes(include=[np.number])

In [24]:
train_features, test_features, train_target, test_target = train_test_split(features, target, test_size=0.2, random_state=21)

loo = model_selection.LeaveOneOut()
ss = model_selection.ShuffleSplit(test_size=0.25)
algorithm = KNeighborsRegressor()

scores1 = model_selection.cross_val_score(algorithm, train_features, train_target, cv=5, scoring="neg_mean_squared_error")
scores2 = model_selection.cross_val_score(algorithm, train_features, train_target, cv=ss, scoring="neg_mean_squared_error")
scores3 = model_selection.cross_val_score(algorithm, train_features, train_target, cv=loo, scoring="neg_mean_squared_error")

model = algorithm.fit(train_features, train_target)
predictions = algorithm.predict(test_features)

rmse = np.sqrt(mean_squared_error(test_target, predictions))

In [25]:
print("Scores sa cv=5", scores1.mean())
print("Scores sa Shuffle Split", scores2.mean())
print("Scores sa Leave One Out", scores3.mean())
print("RMSE", rmse)

Scores sa cv=5 -94912786920.2254
Scores sa Shuffle Split -100743947070.41255
Scores sa Leave One Out -83442347498.45749
RMSE 295562.1522712882
